In [14]:
from DrissionPage import ChromiumPage
import csv
import time
import json
import re

# 创建CSV文件
f = open('NikonZ30_reviews.csv', mode='w', encoding='utf-8-sig', newline='')
csv_writer = csv.DictWriter(f, fieldnames=['昵称','评分','评论','时间','型号配置','购买次数'])
csv_writer.writeheader()

# 启动浏览器
dp = ChromiumPage()

# 访问京东商品页
product_url = 'https://item.jd.com/10055725758017.html?spmTag=YTAyNDAuYjAwMjQ5My5jMDAwMDQwMjcuMiUyM3NrdV9jYXJk'
dp.get(product_url)
print(f"正在访问: {product_url}")

# 给足够时间让你手动登录
print("\n" + "="*50)
print("请在打开的浏览器中手动登录京东账号")
print("你有30秒时间完成登录...")
print("="*50 + "\n")

# 等待30秒让你手动登录
for i in range(30, 0, -1):
    print(f"还剩 {i} 秒...", end='\r')
    time.sleep(1)
print("\n继续执行...")

# 检查是否还在登录页
if 'login' in dp.url or 'passport' in dp.url:
    print("检测到仍在登录页，再多等20秒...")
    time.sleep(20)

print(f"当前页面标题: {dp.title}")
print(f"当前URL: {dp.url}")

# 从URL中提取商品ID
product_id_match = re.search(r'/(\d+)\.html', product_url)
if product_id_match:
    product_id = product_id_match.group(1)
else:
    product_id = '100028098812'  # 默认ID

# 直接使用API爬取评论（不需要点击按钮）
print("\n开始爬取评论...")

page = 0
max_pages = 10
while page < max_pages:
    print(f"\n--- 正在爬取第 {page+1} 页 ---")

    # 京东评论API
    api_url = f'https://club.jd.com/comment/productPageComments.action?productId={product_id}&score=0&sortType=5&page={page}&pageSize=10'

    try:
        # 直接访问API
        dp.get(api_url)
        time.sleep(2)

        # 获取JSON数据
        pre_elem = dp.ele('tag:pre')
        if pre_elem:
            data = json.loads(pre_elem.text)
            comments = data.get('comments', [])

            if not comments:
                print("没有更多评论了")
                break

            print(f"本页找到 {len(comments)} 条评论")

            for comment in comments:
                try:
                    nickname = comment.get('nickname', '匿名用户')
                    score = comment.get('score', '5')
                    content = comment.get('content', '')
                    create_time = comment.get('creationTime', '')

                    color = comment.get('productColor', '')
                    size = comment.get('productSize', '')
                    variant = f"{color} {size}".strip()

                    user_level = comment.get('userLevelInfo', {})
                    buy_count = user_level.get('buyCount', 1)

                    dit = {
                        '昵称': nickname,
                        '评分': str(score),
                        '评论': content,
                        '时间': create_time,
                        '型号配置': variant,
                        '购买次数': str(buy_count)
                    }

                    csv_writer.writerow(dit)
                    print(f"  已写入: {nickname[:10]} - {score}星")

                except Exception as e:
                    print(f"  处理单条评论出错: {e}")
                    continue

            page += 1
        else:
            print("获取API数据失败")
            break

    except Exception as e:
        print(f"请求API出错: {e}")
        break

f.close()
print(f"\n爬取完成！共爬取 {page} 页评论")
print("数据已保存至: jd_reviews.csv")

正在访问: https://item.jd.com/10055725758017.html?spmTag=YTAyNDAuYjAwMjQ5My5jMDAwMDQwMjcuMiUyM3NrdV9jYXJk

请在打开的浏览器中手动登录京东账号
你有30秒时间完成登录...

还剩 1 秒...
继续执行...
当前页面标题: 尼康（Nikon）【年度爆款相机】Z30入门级微单相机Vlog翻转屏自拍旅游家用高清数码相机拆单机 z30相机 Z30 16-50 f/3.5-6.3未开封套装 标配【送膜+64G卡+相机包+座充+清洁套+腕带】【图片 价格 品牌 报价】-京东
当前URL: https://item.jd.com/10055725758017.html?spmTag=YTAyNDAuYjAwMjQ5My5jMDAwMDQwMjcuMiUyM3NrdV9jYXJk

开始爬取评论...

--- 正在爬取第 1 页 ---
请求API出错: Expecting value: line 1 column 1 (char 0)

爬取完成！共爬取 0 页评论
数据已保存至: jd_reviews.csv


合并文件

In [ ]:
import pandas as pd

# 读取两个CSV文件
df1 = pd.read_csv('3Cameras.csv')
df2 = pd.read_csv('Fuji.csv')

# 合并（自动按列名对齐）
df_combined = pd.concat([df1, df2], ignore_index=True)

# 保存合并后的文件
df_combined.to_csv('Cameras_Combined.csv', index=False, encoding='utf-8-sig')

print(f"文件1行数: {len(df1)}")
print(f"文件2行数: {len(df2)}")
print(f"合并后行数: {len(df_combined)}")